# MOSAIC: From Installation to a Playable SAR Mission

This notebook accompanies Part Three of the MOSAIC tutorial. It builds the same search-and-rescue environment shown in the slides, inspects its observations and camera strategies, and launches the real MOSAIC GUI.

> **Run locally.** The GUI opens in a separate Pygame window. A browser-only or remote notebook without desktop display access cannot show that window.

## What you will do

1. Verify the MOSAIC installation.
2. Build a reproducible SAR environment.
3. Inspect the game state and real camera views.
4. Launch and play the GUI.
5. Change the mission configuration.
6. See where an optional AI advisor connects.

![MOSAIC game interface](assets/gui-screenshot.png)

## 1. Verify the installation

Run this notebook from the virtual environment used to install MOSAIC.

In [ ]:
from importlib.metadata import version

import matplotlib.pyplot as plt
import pygame

from mosaic.core.camera import AgentConeCamera, AgentFOVCamera, EdgeFollowCamera
from mosaic.gui.main import SAREnvGUI
from mosaic.sar.env import build_sar_env
from mosaic.sar.placers import LavaPlacer, LockedRoomPlacer, VictimPlacer

print(f"MOSAIC: {version('mosaic')}")
print(f"Pygame: {pygame.version.ver}")
print("Imports ready")

## 2. MOSAIC building blocks

The notebook uses the same four blocks introduced in the presentation:

| Block | Responsibility |
|---|---|
| Gymnasium + MiniGrid | Episode lifecycle and grid-world foundation |
| SAR environment | Rooms, victims, hazards, doors, keys, actions, and observations |
| MOSAIC GUI | Human input, game view, mission information, chat, and event feedback |
| Optional `LLMClient` | Text advice requested by the player |

The helper below keeps mission construction in one place, making each later variation easy to compare.

In [ ]:
import random


def make_environment(
    *,
    seed=11,
    num_rows=2,
    num_cols=2,
    room_size=8,
    victims_per_room=1,
    lava_per_room=1,
    locked_room_prob=0.25,
    camera_strategy=None,
):
    # Current placers use Python's random module, while the environment
    # also accepts a Gymnasium seed. Setting both makes this tutorial repeatable.
    random.seed(seed)

    camera_options = {}
    if camera_strategy is not None:
        camera_options["camera_strategy"] = camera_strategy

    return build_sar_env(
        screen_size=720,
        num_rows=num_rows,
        num_cols=num_cols,
        room_size=room_size,
        victim_placer=VictimPlacer(num_real_victims=victims_per_room),
        lava_placer=LavaPlacer(lava_per_room=lava_per_room),
        locked_room_placer=LockedRoomPlacer(locked_room_prob=locked_room_prob),
        **camera_options,
    )

## 3. Build and inspect a mission

A 2 × 2 layout creates four rooms. `victims_per_room=1` therefore produces four rescue targets in total.

In [ ]:
SEED = 11
env = make_environment(seed=SEED)
observation, reset_info = env.reset(seed=SEED)

print("Mission status:", env.get_mission_status())
print("Agent position:", tuple(int(value) for value in env.agent_pos))
print("Observation keys:", sorted(observation.keys()))

In [ ]:
frame = env.render()

plt.figure(figsize=(6, 6))
plt.imshow(frame)
plt.title("Initial SAR observation")
plt.axis("off")
plt.show()

### Game elements

- The **red triangle** is the player and shows its facing direction.
- **Orange tiles** are lava hazards.
- Colored **doors and keys** control access between rooms.
- A **real victim** has a symmetric cross shape and counts toward the mission.
- A **decoy victim** has an offset cross shape and penalizes a mistaken rescue when a custom placer includes decoys.

![Real victims on the top row and decoy victims on the bottom row](assets/victims.png)

## 4. Compare the camera strategies

All three panels below render the same environment state. Only the camera strategy changes. Depending on the initial agent position, the moving and current-room views may overlap closely.

In [ ]:
camera_views = [
    ("Moving viewport", EdgeFollowCamera()),
    ("Current room", AgentFOVCamera()),
    ("Forward visibility", AgentConeCamera()),
]

figure, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, (title, camera) in zip(axes, camera_views):
    env.switch_camera(camera)
    axis.imshow(env.render())
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()
env.switch_camera(EdgeFollowCamera())

## 5. Use the Gymnasium step interface

MOSAIC remains a Gymnasium environment underneath the GUI. This safe example turns the agent left once and inspects the returned transition.

In [ ]:
from minigrid.core.actions import Actions

observation, reward, terminated, truncated, step_info = env.step(Actions.left)
print("Reward:", reward)
print("Terminated:", terminated, "Truncated:", truncated)
print("Events:", step_info.get("events", []))

## 6. Launch and play

Running the next cell opens the MOSAIC interface in a separate window. The cell remains active until that window closes.

| Key | Action |
|---|---|
| `↑` | Move forward |
| `←` / `→` | Turn |
| `Space` | Open a door |
| `Tab` | Rescue or pick up |
| `Left Shift` | Drop the carried key |
| `Alt` | Request advisor guidance |
| `Backspace` | Restart |
| `F11` | Toggle fullscreen |
| `Esc` | Quit |

In [ ]:
play_env = make_environment(seed=SEED)
gui = SAREnvGUI(
    play_env,
    config={"fullscreen": False, "max_time": 3},
)
gui.run()

## 7. Change the task

The next example changes several constructor values without changing the GUI code:

- 3 × 3 rooms instead of 2 × 2
- two lava tiles per room
- two victims per room
- the full current-room camera

> Try changing **one value at a time** first. This makes its effect easier to understand.

In [ ]:
custom_env = make_environment(
    seed=SEED,
    num_rows=3,
    num_cols=3,
    victims_per_room=2,
    lava_per_room=2,
    locked_room_prob=0.35,
    camera_strategy=AgentFOVCamera(),
)
custom_gui = SAREnvGUI(
    custom_env,
    config={"fullscreen": False, "max_time": 5},
)
custom_gui.run()

## 8. Optional AI advisor

MOSAIC uses a provider-independent `LLMClient` interface. The environment does not need to know which model or provider supplies advice. The default dummy client keeps the game usable without an API key.

When a configured client is available, the connection is one constructor argument:

In [ ]:
# your_client = ...  # Configure an LLMClient using the MOSAIC documentation.
# advisor_env = make_environment(seed=SEED)
# SAREnvGUI(advisor_env, llm_client=your_client).run()

## 9. What to explore next

Choose one small extension:

- **Task world:** change a placer or rescue action.
- **Observation:** choose or implement a camera strategy.
- **Human interface:** customize input, panels, or event feedback.
- **AI support:** inject a client, prompt builder, or response processor.

Useful links: [MOSAIC documentation](https://ihuman-lab.github.io/mosaic/) · [MOSAIC repository](https://github.com/iHuman-Lab/mosaic)

In [ ]:
for variable_name in ("env", "play_env", "custom_env", "advisor_env"):
    candidate = globals().get(variable_name)
    if candidate is not None:
        candidate.close()
pygame.quit()
print("MOSAIC environments closed")